## Playground Series S6E3: Customer Churn Prediction
### Overview

This notebook predicts customer churn for a telecommunications company using a **synthetic version** of the IBM Telco Customer Churn dataset (Playground Series, Season 6, Episode 3).

**Goal:** Predict the probability that a customer will churn (`Churn` column) using demographic and service features. Submissions are evaluated on **ROC-AUC**.

### Strategy

Synthetic datasets can contain noise or label errors introduced during generation. To counter this, we use a **Hybrid Strategy**:

1. **Data Augmentation:** Merge the synthetic competition data with the original IBM dataset to give the model real-world ground truth examples.
2. **Source Tracking:** Add an `is_synthetic` flag so the model can learn to weight rows differently depending on their origin.
3. **Real-World Signal:** For each synthetic row, find its closest match in the original dataset and borrow that customer's churn label as an extra feature (`original_churn_signal`).

**Models:** LightGBM / XGBoost  
**Validation:** Stratified K-Fold Cross-Validation

---
# 1. Setup

## 1.1 Import Libraries

In [1]:
# Standard Library Imports
import os
from subprocess import check_output
import warnings

# Third-Party Library Imports
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from scipy.stats import rankdata
from xgboost import XGBClassifier

# Scikit-Learn Imports (Specific Sub-modules)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import NearestNeighbors

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## 1.2 Load Data

In [2]:
print(check_output(["ls", "../input"]).decode("utf8"))
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

competitions
datasets

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [3]:
# Load the competition data
train = pd.read_csv('../input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('../input/competitions/playground-series-s6e3/test.csv')
sample_submission = pd.read_csv('../input/competitions/playground-series-s6e3/sample_submission.csv')

# Load the original IBM dataset — we'll merge it with train later
original = pd.read_csv('../input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Train shape:    {train.shape}")
print(f"Test shape:     {test.shape}")
print(f"Original shape: {original.shape}")

Train shape:    (594194, 21)
Test shape:     (254655, 20)
Original shape: (7043, 21)


In [4]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
test.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [6]:
original.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1.3 Pre-Merge Alignment

Before merging, both datasets need to speak the same language:
- **Column names** — drop identifiers that won't be used as features
- **Data types** — all shared columns must have the same dtype
- **Target labels** — `Churn` must be encoded the same way in both

In [7]:
# Compare column data types between train and original to catch any mismatches
# before merging
pd.concat([train.dtypes, original.dtypes], axis=1, keys=['train', 'original'])

,train,original
id,int64,NaN
gender,object,object
SeniorCitizen,int64,int64
Partner,object,object
Dependents,object,object
tenure,int64,int64
PhoneService,object,object
MultipleLines,object,object
InternetService,object,object
OnlineSecurity,object,object


Two issues to fix:
- `customerID` in original vs `id` in train; both are identifiers with no predictive value, so we drop both.
- `TotalCharges` is `float64` in train but `object` in original; needs to be cast to `float64` in both.

In [8]:
# Drop identifier columns — not needed as features
train    = train.drop(columns=['id'])
original = original.drop(columns=['customerID'])

# Cast TotalCharges to float in original (it was stored as text)
# errors='coerce' turns any non-numeric entries into NaN
original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
print(f"TotalCharges NaN count in original: {original['TotalCharges'].isna().sum()}")

TotalCharges NaN count in original: 11


In [9]:
# The 11 NaN values are new customers who haven't been billed yet — fill with 0
original['TotalCharges'] = original['TotalCharges'].fillna(0)
train['TotalCharges']    = pd.to_numeric(train['TotalCharges'], errors='coerce').fillna(0)
test['TotalCharges']     = pd.to_numeric(test['TotalCharges'],  errors='coerce').fillna(0)

## 1.4 Anchoring Synthetic Rows to Real-World Churn Signals

Our synthetic data is generated by AI, so some labels may not reflect real-world behaviour. To address this, we borrow a "ground truth hint" from the original dataset for every synthetic row.

**How it works:**

We use a KNN model (an algorithm that finds the most similar rows) to match each synthetic row to its nearest real customer. Similarity is measured using `tenure`, `MonthlyCharges`, and `TotalCharges` — the three numerical features that best describe a customer's financial profile (and the only ones we have actually ^_^)

For each match, we:
- Copy the real customer's `Churn` label into a new feature: `original_churn_signal`
- Store how close the match was in `match_dist` — the smaller the distance, the more trustworthy the signal

We're not replacing any label, just adding extra information. The model learns on its own how much to trust this signal based on the match distance.

In [10]:
# These 3 features define the space we use to measure similarity between customers
features = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Place every original customer in that space so we can find the closest match later
nn = NearestNeighbors(n_neighbors=5).fit(original[features])

# Borrow that real customer's Churn label as a ground truth hint
original['Churn'] = original['Churn'].str.strip().str.lower().map({'yes': 1, 'no': 0})
train['Churn'] = train['Churn'].str.strip().str.lower().map({'yes': 1, 'no': 0})

# For each synthetic train row, find the closest real customer
# Store the distance; the smaller it is, the more trustworthy the signal

distances_tr, indices_tr = nn.kneighbors(train[features])
train['original_churn_signal'] = original.iloc[indices_tr.flatten()]['Churn'].values.reshape(-1, 5).mean(axis=1)
train['match_dist'] = distances_tr.mean(axis=1)

distances_ts, indices_ts = nn.kneighbors(test[features])
test['original_churn_signal'] = original.iloc[indices_ts.flatten()]['Churn'].values.reshape(-1, 5).mean(axis=1)
test['match_dist'] = distances_ts.mean(axis=1)

## 1.5 Merging Synthetic and Real Data

In [11]:
# Tag each row so the model knows whether it came from the synthetic or real dataset
train['is_synthetic']    = 1
test['is_synthetic']     = 1
original['is_synthetic'] = 0

# Merge synthetic train with the real original dataset
# Original rows won't have 'original_churn_signal' or 'match_dist',
# so we fill them with sensible defaults after merging
train_full = pd.concat([train, original], axis=0).reset_index(drop=True)
train_full['original_churn_signal'] = train_full['original_churn_signal'].fillna(train_full['Churn'])
train_full['match_dist']            = train_full['match_dist'].fillna(0)

print(f"Final training shape: {train_full.shape}")

Final training shape: (601237, 23)


---
# 2. Diagnose Data Issues

We run a fixed set of checks on `train_full` to confirm the merge went cleanly and catch any remaining issues before modelling.

## 2.1 Basic Overview

In [12]:
# Shape, column names, and dtypes
train_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 601237 entries, 0 to 601236
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   gender                 601237 non-null  object 
 1   SeniorCitizen          601237 non-null  int64  
 2   Partner                601237 non-null  object 
 3   Dependents             601237 non-null  object 
 4   tenure                 601237 non-null  int64  
 5   PhoneService           601237 non-null  object 
 6   MultipleLines          601237 non-null  object 
 7   InternetService        601237 non-null  object 
 8   OnlineSecurity         601237 non-null  object 
 9   OnlineBackup           601237 non-null  object 
 10  DeviceProtection       601237 non-null  object 
 11  TechSupport            601237 non-null  object 
 12  StreamingTV            601237 non-null  object 
 13  StreamingMovies        601237 non-null  object 
 14  Contract               601237 non-nu

No missing data, and all dtypes look correct after our earlier cleaning steps.

In [13]:
# Statistical summary for numerical columns
train_full.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,original_churn_signal,match_dist,is_synthetic
count,601237.000000,601237.000000,601237.000000,601237.000000,601237.000000,601237.000000,601237.000000,601237.000000
mean,0.114665,36.527986,65.853285,2491.862692,0.225678,0.247361,6.582215,0.988286
std,0.318618,25.060161,31.056376,2353.026339,0.418028,0.285473,4.316902,0.107596
min,0.000000,0.000000,18.250000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,12.000000,29.900000,639.150000,0.000000,0.000000,3.635324,1.000000
50%,0.000000,35.000000,74.100000,1432.650000,0.000000,0.200000,6.267037,1.000000
75%,0.000000,62.000000,90.800000,4263.400000,0.000000,0.400000,8.858774,1.000000
max,1.000000,72.000000,118.750000,8684.800000,1.000000,1.000000,88.986817,1.000000


**Observations:**
- `SeniorCitizen` is binary (0/1); don't treat it as continuous, "average SeniorCitizen" is meaningless.
- `TotalCharges` is likely right-skewed; may benefit from log transformation.
- `tenure` minimum is 0 after merging; we'll watch for this in feature engineering (avoid division by tenure!).

In [14]:
# Statistical summary for categorical columns
train_full.describe(include='object')

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
count,601237,601237,601237,601237,601237,601237,601237,601237,601237,601237,601237,601237,601237,601237,601237
unique,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4
top,Female,Yes,No,Yes,No,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check
freq,302226,312956,419295,564254,286774,275482,292972,253171,250472,292044,243008,244167,302793,369750,217737


**Observations:**
- `Churn` is imbalanced — roughly 77% No — but not severely so.
- `PhoneService` is heavily skewed toward "Yes" (~94%).
- `gender`, `Partner`, `Dependents`, `PhoneService`, and `PaperlessBilling` are all binary Yes/No.
- **Typical Customer:** Female, has partner, no dependents, has phone, no multiple lines, fiber optic internet, month-to-month contract, paperless billing, electronic check payment and hasn't churned!

## 2.2 Missing or Null Entries
Check for NaNs, empty strings, and whitespace-only values across all columns.

In [15]:
# Count and percentage of missing values per column
missing     = train_full.isnull().sum()
missing_pct = (missing / len(train_full)) * 100
missing_df  = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})

non_zero = missing_df[missing_df['Count'] > 0].sort_values('Percentage', ascending=False)
if non_zero.empty:
    print("No missing values found. But we already know that!")
else:
    display(non_zero)

# Check for empty strings or whitespace-only values in categorical columns
obj_cols   = train_full.select_dtypes(include='object').columns
empty_found = False
for col in obj_cols:
    empty_count = train_full[col].apply(lambda x: str(x).strip() == '').sum()
    if empty_count > 0:
        print(f"{col}: {empty_count} empty/whitespace-only values")
        empty_found = True
if not empty_found:
    print("No empty strings or whitespace-only values found.")

No missing values found. But we already know that!
No empty strings or whitespace-only values found.


## 2.3 Negative and Out-of-Bounds Values
Check numerical columns for values that don't make sense.

In [16]:
# Check for negative values in numerical columns
num_cols    = train_full.select_dtypes(include=np.number).columns
neg_found   = False
for col in num_cols:
    neg_count = (train_full[col] < 0).sum()
    if neg_count > 0:
        print(f"{col}: {neg_count} negative values")
        neg_found = True
if not neg_found:
    print("No negative values found.")


# Min and max per numerical column — transposed so features are rows, easier to scan
train_full[num_cols].agg(['min', 'max']).T

No negative values found.


,min,max
SeniorCitizen,0.00,1.000000
tenure,0.00,72.000000
MonthlyCharges,18.25,118.750000
TotalCharges,0.00,8684.800000
Churn,0.00,1.000000
original_churn_signal,0.00,1.000000
match_dist,0.00,88.986817
is_synthetic,0.00,1.000000


**Observations:**
- No negative or out-of-bounds values.
- `tenure` min is 0; new customers with no completed months. Let's just avoid / by tenure 

## 2.4 Placeholder Values
Look for placeholder values like `-1`, `999`, or `'unknown'` that may be masking missing data.

In [ ]:
# Check for common numeric placeholders
placeholders = [-1, 999, 9999, -999]
ph_found     = False
for col in num_cols:
    for p in placeholders:
        count = (train_full[col] == p).sum()
        if count > 0:
            pct = count / len(train_full) * 100
            print(f"{col} == {p}: {count} occurrences ({pct:.2f}%)")
            ph_found = True
if not ph_found:
    print("No numeric placeholders found.")

# Check for placeholder strings in categorical columns
placeholder_strings = ['unknown', 'n/a', 'na', 'none', 'null', 'missing', '', ' ']
ph_str_found        = False
for col in obj_cols:
    for p in placeholder_strings:
        count = train_full[col].str.lower().str.strip().eq(p).sum()
        if count > 0:
            print(f"{col} == '{p}': {count} occurrences")
            ph_str_found = True
if not ph_str_found:
    print("No placeholder strings found.")

No numeric placeholders found.


## 2.5 Logical Consistency
Check for logical contradictions in customers' data (for instance, a customer with no internet but has internet add-ons, or no phone but has multiplelines!)

In [ ]:
# Customers with no internet service but with internet-dependent add-ons
internet_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies']
inc_found = False
for col in internet_services:
    count = ((train_full['InternetService'] == 'No') & (train_full[col] == 'Yes')).sum()
    if count > 0:
        print(f"{col}: {count} inconsistent rows")
        inc_found = True

# Customers with no phone service but MultipleLines enabled
count = ((train_full['PhoneService'] == 'No') & (train_full['MultipleLines'] == 'Yes')).sum()
if count > 0:
    print(f"MultipleLines: {count} inconsistent rows")
    inc_found = True

if not inc_found:
    print("No logical inconsistencies found.")

---
# 3. Data Quality & Integrity Checks

## 3.1 Duplicate Records

In [ ]:
# Check for fully duplicate rows
dup_count = train_full.duplicated().sum()
print(f"Duplicate rows: {dup_count} ({dup_count / len(train_full) * 100:.2f}%)")

Identified 22 duplicate entries. Given the massive dataset size (600k+ rows), these are statistically insignificant and can be safely ignored or removed!

## 3.2 Cardinality Check

In [ ]:
# Unique value counts per column; helps spot unexpected high-cardinality columns
cat_df = train_full.select_dtypes(include=['object', 'category'])

cardinality    = cat_df.nunique().sort_values(ascending=False)
cardinality_df = pd.DataFrame({'Unique Values': cardinality, 'Dtype': train_full.dtypes[cardinality.index]})
cardinality_df

**Observations:**
- All categoricals have low cardinality (2–4 unique values); straightforward to encode, no issues.

## 3.4 Train/Test Schema Alignment
Confirm train and test have the same columns (excluding the target).

In [ ]:
# Compare train_full and test columns
train_cols = set(train_full.columns) - {'Churn', 'original_churn_signal', 'match_dist', 'is_synthetic'}
test_cols  = set(test.columns)

print(f"Only in train: {train_cols - test_cols}")
print(f"Only in test:  {test_cols  - train_cols}")

In [ ]:
# Confirm dtypes match between train and test for shared columns
shared_cols       = list(train_cols & test_cols)
dtype_comparison  = pd.DataFrame({'Train': train_full[shared_cols].dtypes, 'Test': test[shared_cols].dtypes})
mismatches        = dtype_comparison[dtype_comparison['Train'] != dtype_comparison['Test']]
if mismatches.empty:
    print("All shared columns have matching dtypes.")
else:
    display(mismatches)

---
# 4. Distribution & Statistical Checks

## 4.1 Target Distribution

In [ ]:
# Target variable distribution
churn_dist = train_full['Churn'].value_counts()
print(churn_dist)

In [ ]:
plt.figure(figsize=(5, 3))
ax = churn_dist.plot(kind='bar', edgecolor='black')

total = churn_dist.sum()
for p in ax.patches:
    pct = f"{100 * p.get_height() / total:.1f}%"
    ax.annotate(pct, (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')

plt.title('Churn Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Observations:**
- ~77.4% No vs ~22.6% Yes. Imbalanced but not severely.
- Tree-based models (LightGBM, XGBoost) handle this well without resampling.
- Worth trying `scale_pos_weight` during tuning to see if it helps.

## 4.2 Numerical Feature Distributions

In [ ]:
# Define numerical columns to plot (exclude binary flags and engineered signals)
num_cols_plot = ['tenure', 'MonthlyCharges', 'TotalCharges']

ncols = 2
nrows = (len(num_cols_plot) + 1) // 2

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 3 * nrows))
axes = axes.flatten()

for i, col in enumerate(num_cols_plot):
    train_full[col].hist(bins=50, ax=axes[i], edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')

for j in range(len(num_cols_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('numeric_skew.png', dpi=300, bbox_inches='tight')
plt.show()

**Observations:**
- `tenure`: Bimodal; customers either leave early (1–5 months) or stay long-term (70–72 months).
- `MonthlyCharges`: Right-skewed, with a cluster of low-charge customers likely on basic plans.
- `TotalCharges`: Heavily right-skewed, expected since it's roughly `tenure × MonthlyCharges`. Log transformation may help.

## 4.3 Categorical Feature Distributions (Univariate)

In [ ]:
# Bar plots for all categorical features (except for original_churn_signal)

ncols = 2
nrows = (len(obj_cols) + 1) // 2

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(obj_cols):
    counts = train_full[col].value_counts()
    counts.plot(kind='bar', ax=axes[i], edgecolor='black', color='steelblue')
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)
    total = counts.sum()
    for p in axes[i].patches:
        pct = f"{100 * p.get_height() / total:.1f}%"
        axes[i].annotate(pct, (p.get_x() + p.get_width() / 2, p.get_height()),
                         ha='center', va='bottom', fontsize=9)

for j in range(len(obj_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('Univariate.png', dpi=300, bbox_inches='tight')
plt.show()

**Observations:**
- `gender` is almost balanced; not gonna be strong predictor.
- `Partner` and `Dependents` little right-skewed but noting severe.
- `PhoneService` is highly right-skewed → strong predictor.
- `MultipleLines` is nearly balanced between single ~47% and multiple lines ~46%, with a small 6.2% group having no phone service; consistent with the PhoneService skew.
- All internet add-on services (OnlineSecurity, OnlineBackup, etc.) show a consistent "No internet service" slice at ~23%; consistent with the `InternetService` skew. Other than that, all are right-skewed.
- `Contract` is dominated by Month-to-month (50.4%).
- Most customers are on `PaperlessBilling` plan.
- `PaymentMethod` is distributed almost evenly across it's methods, with Electronic check slightly leading.
- Several features (`MultipleLines, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies`) contain a "No internet service" or "No phone service" category that means the same thing as "No". We will collapse these into "No" before encoding to reduce unnecessary cardinality.

## 4.4 Skewness

In [ ]:
# Skewness of numerical columns
skew = train_full[num_cols_plot].skew().sort_values(ascending=False)
print("Skewness:\n", skew)

**Skewness confirms and corrects our visual observations:**

- `TotalCharges` (0.91): Confirms strong right skew.
- `tenure` (0.065): Near zero skew, but the histogram showed a bimodal shape. That's not a contradiction; skewness only measures whether values lean left or right, not how many peaks exist. The two peaks at opposite ends happen to cancel each other out.
- `MonthlyCharges` (-0.29): Visually looks right-skewed due to the spike at 20, but the skewness value reveals a slight left skew; the bulk of customers sit in the 60–120 range, and the ~20 cluster pulls the mean leftward. Another reminder not to rely on visuals alone.

## 4.5 Outlier Detection

In [ ]:
# IQR-based outlier detection
outlier_summary = {}
for col in num_cols_plot:
    Q1 = train_full[col].quantile(0.25)
    Q3 = train_full[col].quantile(0.75)
    IQR = Q3 - Q1
    lower   = Q1 - 1.5 * IQR
    upper   = Q3 + 1.5 * IQR
    outliers = ((train_full[col] < lower) | (train_full[col] > upper)).sum()
    if outliers > 0:
        outlier_summary[col] = {
            'Count': outliers,
            'Pct':   round(outliers / len(train_full) * 100, 2),
            'Lower': lower,
            'Upper': upper
        }

if outlier_summary:
    pd.DataFrame(outlier_summary).T.sort_values('Pct', ascending=False)
else:
    print("No outliers detected.")

---
# 5. Feature Analysis

In [ ]:
corr = train_full[num_cols_plot].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(6, 4))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='YlGnBu', center=0)
plt.title('Feature Correlation Matrix')
plt.tick_params(axis='both', labelsize=8)
plt.tight_layout()
plt.show()

**Observations**
- `TotalCharges` correlates very well with both of `tenure` and `MonthlyCharges` which only makes sense since Total Charges is calculated out of these other two variables.
- `tenure` and `MonthlyCharges` do not correlate well; indicating that a long-tenure customer is probably not on a high-cost plan.

## 5.1 Feature vs Target Relationship

In [ ]:
# How each numerical feature distributes across Churn = 0 vs 1
for col in num_cols_plot:
    plt.figure(figsize=(8, 3))
    for label in [0, 1]:
        train_full[train_full['Churn'] == label][col].hist(bins=50, alpha=0.5, label=str(label))
    plt.title(f'{col} by Churn')
    plt.legend(title='Churn')
    plt.tight_layout()
    plt.savefig('featurevstarget.png', dpi=300, bbox_inches='tight')
    plt.show()

**Observation**

Churned customers consistently appear at short tenure and low total charges, confirming they leave early before accumulating significant billing history. `MonthlyCharges` tells a different story; churned customers are more concentrated in the higher charge range (60–100), suggesting that customers on expensive plans are more likely to leave.

### 5.1.1 Catgorical Fatures vs Target
My primary heuristic for evaluating feature importance is simple: a feature is a strong predictor if the churn proportion varies significantly across its categories. If the bars are roughly the same height, the feature likely provides little signal for the model.

In [ ]:
# Collapse "No internet service" and "No phone service" into "No"
# These mean the same thing and adding them as separate categories adds no signal
internet_addons = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

for df in [train_full, test]:
    for col in internet_addons:
        df[col] = df[col].replace('No internet service', 'No')
    df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

In [ ]:
# How each categorical feature splits across Churn = 0 vs 1
cols_to_plot = [col for col in obj_cols if col != 'Churn']
ncols = 2
nrows = (len(cols_to_plot) + 1) // 2

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4 * nrows))
axes = axes.flatten()

colors = plt.cm.YlGnBu([0.4, 7])  # pick two shades from the palette

for i, col in enumerate(cols_to_plot):
    ct = pd.crosstab(train_full[col], train_full['Churn'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=colors)
    axes[i].set_title(f'{col} vs Churn')
    axes[i].set_ylabel('Proportion')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)

for j in range(len(cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('catvstarget.png', dpi=300, bbox_inches='tight')
plt.show()

**Observations**

**Strong Perdictors**

- `Contract`: Month-to-month churns at ~40%, One year and Two year barely churn. **Strongest signal.**
- `InternetService`: Fiber optic has dramatically higher churn than DSL or No internet.
- `OnlineSecurity`, `TechSupport`: No group churns at roughly double the rate of Yes group.

**Moderate predictors**

- `DeviceProtection, OnlineBackup`: same direction as OnlineSecurity but smaller gap.
- `Partner, Dependents`: No group churns more, visible but not dramatic.
- `PaymentMethod`: Electronic check clearly higher than the other three.
- `StreamingTV, StreamingMovies`: very similar bars, marginal difference.

**Weak predictors**

- `PaperlessBilling`: small gap between Yes and No.
- `MultipleLines`: barely any difference after merging.
- `PhoneService`: almost identical bars.
- `gender`: no visible difference at all.

---
# 6. Feature Engineering

In [ ]:
# Before encoding
for df in [train_full, test]:
    df['contract_x_internet'] = df['Contract'].astype(str) + '_' + df['InternetService'].astype(str)

# ORIG_proba features; map real churn rate from original dataset per category
# For each categorical column, we look up the actual churn rate in the real data
# and use it as a feature. This gives the model a direct signal from ground truth.
orig_cat_cols = ['Contract', 'InternetService', 'PaymentMethod',
                 'OnlineSecurity', 'TechSupport', 'PaperlessBilling',
                 'Partner', 'Dependents', 'MultipleLines']

for col in orig_cat_cols:
    prob_map = original.groupby(col)['Churn'].mean()
    train_full[f'orig_proba_{col}'] = train_full[col].map(prob_map).fillna(0.5)
    test[f'orig_proba_{col}']       = test[col].map(prob_map).fillna(0.5)


for df in [train_full, test]:
    # Behavioural   
    df['no_protection_score'] = (df[['OnlineSecurity', 'OnlineBackup', 
                                      'DeviceProtection', 'TechSupport']].eq('No')).sum(axis=1)    
    # Contract risk
    df['mtm_fiber'] = ((df['Contract'] == 'Month-to-month') & 
                        (df['InternetService'] == 'Fiber optic')).astype(int)
    df['autopay_flag'] = (df['PaymentMethod'] != 'Electronic check').astype(int)
    df['has_any_addon'] = (df[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                                'TechSupport', 'StreamingTV', 'StreamingMovies']].eq('Yes')).any(axis=1).astype(int)

orig_proba_cols = [f'orig_proba_{col}' for col in orig_cat_cols]
all_new = orig_proba_cols + ['no_protection_score', 'mtm_fiber', 'autopay_flag', 'has_any_addon']
print("Pre-encoding features — correlation with Churn:")
print(train_full[all_new + ['Churn']].corr()['Churn'].sort_values(ascending=False))

## 6.1 Encode Categoricals

Columns with 2 categories will be encoded with 0/1, other with 3 or 4 categories will be one-hot encoded.

In [ ]:
binary_cols = []
onehot_cols = []

obj_cols = train_full.select_dtypes(include='object').columns
for col in obj_cols:
    n = train_full[col].nunique()
    if n == 2:
        binary_cols.append(col)
    else:
        onehot_cols.append(col)

print(f"Binary columns ({len(binary_cols)}): {binary_cols}")
print(f"One-hot columns ({len(onehot_cols)}): {onehot_cols}")

# Encode binary columns
for df in [train_full, test]:
    for col in binary_cols:
        unique_vals = df[col].unique()
        # Handle Yes/No
        if set(unique_vals) <= {'Yes', 'No'}:
            df[col] = df[col].map({'Yes': 1, 'No': 0})
        # Handle Male/Female
        elif set(unique_vals) <= {'Male', 'Female'}:
            df[col] = df[col].map({'Male': 1, 'Female': 0})

# One-hot encode multi-category columns
train_full = pd.get_dummies(train_full, columns=onehot_cols, drop_first=True)
test       = pd.get_dummies(test,       columns=onehot_cols, drop_first=True)

print(f"\nNew train_full shape: {train_full.shape}")
print(f"New test shape:       {test.shape}")

## 6.2 Create New Features

In the correlation analysis we noticed that `TotalCharges` is basically `tenure` × `MonthlyCharges` ;just the monthly bill accumulated over time. This relationship opens up several useful features:

- `charges_per_month`: dividing `TotalCharges` by `tenure` gives the average monthly spend over the customer's lifetime. If this differs from their current `MonthlyCharges`, it means their plan changed at some point; an upgrade or downgrade that could signal churn risk.
- `charges_deviation`: the raw difference between `TotalCharges` and `tenure` × `MonthlyCharges`. A non-zero value means billing history doesn't match the current plan,  capturing the same signal differently.
- `monthly_to_total_ratio`: current monthly charge as a proportion of lifetime spend. For example, a customer who has paid £1,000 total with a £100 monthly bill has a 10% ratio. A new customer who has only paid £100 total with the same £100 bill has a 100% ratio. The higher this number, the newer the customer — and newer customers are more likely to churn because they haven't committed yet. The model uses this to flag customers who haven't settled in.

From the bivariate analysis, `Contract` and `InternetService` were our two strongest predictors. A customer on `month-to-month` contract with `fiber optic` is likely a very different churn risk than either feature alone suggests, so we also create a combined interaction feature.

Finally, looking at the service columns together: `PhoneService, OnlineSecurity, StreamingTV` etc. A customer with more active services is more invested in the platform. We summarise this as `service_on`.

In [ ]:
for df in [train_full, test]:
    # Service count (must come before price_per_service)
    service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['service_count'] = df[service_cols].sum(axis=1)
    
    # Charges-based
    df['charges_per_month'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], 0)
    df['charges_deviation'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])
    df['monthly_to_total_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['price_per_service'] = df['MonthlyCharges'] / df['service_count'].clip(lower=1)


    # Tenure-based
    tenure_bins = pd.cut(df['tenure'], bins=[-1, 6, 24, 48, 72], labels=[0, 1, 2, 3]).astype(int)
    df['tenure_bin_x_monthly'] = tenure_bins * df['MonthlyCharges']
    df['early_life_flag'] = (df['tenure'] <= 6).astype(int)
    df['tenure_squared'] = df['tenure'] ** 2

post_encode_features = ['charges_per_month', 'charges_deviation', 'monthly_to_total_ratio',
                        'service_count', 'price_per_service', 'tenure_bin_x_monthly',
                        'early_life_flag', 'tenure_squared']
print("Post-encoding features — correlation with Churn:")
print(train_full[post_encode_features + ['Churn']].corr()['Churn'].sort_values(ascending=False))

- `early_life_flag` (0.314) is the strongest post-encoding feature, confirming that customers in their first 6 months are significantly more likely to churn.
- `price_per_service` (0.307) and `monthly_to_total_ratio` (0.247) are both strong, capturing different angles: cost per service and how new the customer is relative to their spend.
- `tenure_squared` (-0.378) is the strongest negative signal — longer-tenured customers churn far less, and the squared term captures the non-linear drop-off in risk.
- `tenure_bin_x_monthly` (-0.187) shows that high-tenure, high-spend customers are sticky.
- `charges_deviation` (0.037) and `service_count` (-0.037) show weak linear correlation, but we keep them as tree-based models can extract value from features that interact well with others even when their standalone correlation is low.

# 7. Model Training

Tree-based boosting models (LightGBM, XGBoost) are well-established as the strongest performers for tabular churn prediction tasks. We skip baseline models like Logistic Regression and Random Forest deliberately; they serve as reference points in general workflows, but given our dataset size (~600k rows) and feature mix, they are unlikely to compete and would add training time without useful insight.

## 7.1 Prepare Features and Target

First, we'll confirm test has the same columns before aligning:

In [ ]:
y = train_full['Churn']
X = train_full.drop(columns = 'Churn')
X_test = test.drop(columns=['id'])

# Align columns — test may be missing some one-hot columns that exist in train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Stratified split to preserve class balance
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

## 7.2 Advanced Models (LightGBM, XGBoost)

We evaluate each model using 5-fold cross-validation; the training data is split into 5 parts, the model trains on 4 and is tested on the remaining 1, repeated 5 times. This gives us 5 AUC scores per model. The mean tells us overall performance and the standard deviation tells us how stable the model is across different subsets of the data.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LightGBM': LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost':  XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42,
                               eval_metric='auc', verbosity=0),
}

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc')
    results[name] = {'Mean AUC': cv_scores.mean(), 'Std': cv_scores.std()}
    print(f"{name}: AUC = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

All three models score virtually identically. We will proceed with an ensemble of all three to squeeze out the best possible AUC.

---
# 8. Model Selection & Evaluation

## 8.1 Compare All Models

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Mean AUC', ascending=False)
display(results_df)

## 8.2  Hyperparameter Tuning & Stacking Ensemble

Default hyperparameters got all three models to ~0.917 AUC, but there is likely more performance to extract. We will use Optuna to search for better hyperparameters for XGBoost and LightGBM, then combine both tuned models using a stacking ensemble.

**How it works:**

Optuna runs 15 trials per model, each time testing a different combination of hyperparameters (learning rate, tree depth, regularisation, etc.). Early stopping halts training when the validation score stops improving, so we don't waste time on unnecessary rounds.

The best hyperparameters from each model are then used to build a stacking ensemble: both models generate out-of-fold predictions, and a Logistic Regression meta-learner learns the optimal way to combine them.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Hold out 20% for early stopping — separate from the CV split used earlier
X_tune, X_eval, y_tune, y_eval = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Optuna objective for XGBoost ---
# Each trial samples a set of hyperparameters, trains with early stopping,
# and returns the best AUC on the held-out eval set
def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=2000,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        random_state=42, early_stopping_rounds=50, eval_metric='auc'
    )
    model.fit(X_tune, y_tune, eval_set=[(X_eval, y_eval)], verbose=False)
    return model.best_score

# --- Optuna objective for LightGBM ---
# Same idea, but optimises binary logloss (lower is better)
def objective_lgbm(trial):
    model = LGBMClassifier(
        n_estimators=2000,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 20, 150),
        max_depth=trial.suggest_int('max_depth', 3, 12),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        random_state=42, verbose=-1
    )
    model.fit(X_tune, y_tune, eval_set=[(X_eval, y_eval)],
              callbacks=[early_stopping(50), log_evaluation(0)])
    return model.best_score_['valid_0']['binary_logloss']

In [ ]:
# --- Run 15 trials per model ---
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=15)
print(f"XGBoost best AUC: {study_xgb.best_value:.4f}")
print(f"XGBoost best params: {study_xgb.best_params}")

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=15)
print(f"LightGBM best logloss: {study_lgbm.best_value:.4f}")
print(f"LightGBM best params: {study_lgbm.best_params}")

In [ ]:
# --- Stacking Ensemble ---
# Use the best hyperparameters from tuning to build the final models
# StackingClassifier generates out-of-fold predictions from both base models,
# then a Logistic Regression learns how to best combine them
estimators = [
    ('xgb', XGBClassifier(**study_xgb.best_params, n_estimators=2000, random_state=42, eval_metric='auc')),
    ('lgbm', LGBMClassifier(**study_lgbm.best_params, n_estimators=2000, random_state=42, verbose=-1)),
]

final_stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method='predict_proba'
)

In [ ]:
final_stack.fit(X, y)
ensemble_preds = final_stack.predict_proba(X_test)[:, 1]
print(f"Ensemble predictions ready — shape: {ensemble_preds.shape}")

In [ ]:
# Check stacking performance on validation set
val_preds = final_stack.predict_proba(X_val)[:, 1]
print(f"Stacking Ensemble Validation AUC: {roc_auc_score(y_val, val_preds):.4f}")

In [ ]:
print(f"XGBoost best params: {study_xgb.best_params}")
print(f"LightGBM best params: {study_lgbm.best_params}")

---
# 9. Generate Submission

Since `sample_submission` already has the correct structure Kaggle expects; the right column names and row order, we copy it and simply replace the `Churn` column with our predicted probabilities.

In [ ]:
# Save ensemble predictions to submission file
submission = sample_submission.copy()
submission['Churn'] = ensemble_preds
submission.to_csv('submission_ensemble.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"Prediction range: [{ensemble_preds.min():.4f}, {ensemble_preds.max():.4f}]")
print(f"Prediction mean:  {ensemble_preds.mean():.4f}")
submission.head()

In [ ]:
# --- Alternative: Rank Averaging Ensemble ---
# Simpler than stacking, often generalises better to unseen test data
lgbm_model = LGBMClassifier(**study_lgbm.best_params, n_estimators=2000, random_state=42, verbose=-1)
xgb_model = XGBClassifier(**study_xgb.best_params, n_estimators=2000, random_state=42, eval_metric='auc')

lgbm_model.fit(X, y)
xgb_model.fit(X, y)

lgbm_preds = lgbm_model.predict_proba(X_test)[:, 1]
xgb_preds = xgb_model.predict_proba(X_test)[:, 1

val_lgbm = lgbm_model.predict_proba(X_val)[:, 1]
val_xgb = xgb_model.predict_proba(X_val)[:, 1]
val_rank = (rankdata(val_lgbm) + rankdata(val_xgb)) / (2 * len(val_lgbm))
print(f"Rank Avg Validation AUC: {roc_auc_score(y_val, val_rank):.4f}")

In [ ]:
ensemble_preds_rank = (rankdata(lgbm_preds) + rankdata(xgb_preds)) / (2 * len(lgbm_preds))

# Save as a separate submission to compare against stacking
submission_rank = sample_submission.copy()
submission_rank['Churn'] = ensemble_preds_rank
submission_rank.to_csv('submission_rank_avg.csv', index=False)
print(f"Prediction range: [{ensemble_preds_rank.min():.4f}, {ensemble_preds_rank.max():.4f}]")

**Final submission:** Rank averaging outperformed stacking on the leaderboard (0.91438 vs 0.91433), so we use rank averaging as our final ensemble strategy.